# Voice Command Pipeline v2: SraVaani → script-invariant parser (RTX A500 laptop)

**Target:** NVIDIA RTX A500 Laptop GPU, Intel i7-1355U, 16 GB RAM. There is no motor yet; the final output is a validated intent.

### What changed from the previous notebook, and why

| Old | New | Why |
|---|---|---|
| CPU-only PyTorch wheel | CUDA wheel, GPU check | The A500 was never used. |
| RMS energy endpointing, 1.2 s silence | Silero VAD with hysteresis, 350 ms hangover | Roughly 0.85 s less waiting per command; not fooled by steady noise. |
| WAV written to disk per command | In-memory decoding (auto-falls back to the file path) | Removes I/O; the documented path is still available for comparison. |
| Blocking loop (deaf while inferring) | Audio thread + ASR worker thread | Listening, and therefore STOP, continues during inference. |
| Laya decides the command | Deterministic lexicon on **script-canonicalised** text | SraVaani sometimes outputs the right sounds in the wrong script; folding all Indic scripts into one romanised form makes that harmless. Sub-millisecond, auditable. |
| Stop keywords in Latin and Devanagari only | Stop works in any Indic script, Hinglish, and negated motion | `রুকো`, `प्लीज स्टॉप` and "aage mat jao" all stop. |
| Wake word required for everything | Wake word arms one *motion* command; **STOP never needs it** | A stop must not depend on a wake phrase. |
| No telemetry | Every segment's audio, transcript, script histogram, intent and stage timings are logged | Live use produces your evaluation and fine-tuning data. |
| — | Optional script-constrained CTC decoding (NeMo) | Enforces Hindi + English *inside* the decoder and measures language drift. |

### Flow

```
mic (int16, 32 ms blocks)
 ├─ openWakeWord (CPU) ──► arms ONE motion command for ARM_WINDOW_S
 └─ Silero VAD (CPU) ──► segment (pre-roll + speech, hangover trimmed)
                              │  queue (audio loop never blocks)
                              ▼
                   ASR worker: SraVaani on GPU (FP16)
                              ▼
     script analysis → fold every Indic script to Devanagari → romanise → phonetic keys
                              ▼
     lexicon parser: STOP > negated motion (→ STOP) > single motion > ambiguous/none → OTHER
                              ▼
     gate: STOP always | motion only if armed | other ignored  ──►  sink (console now; serial/CAN later)
```

**Run order:** Section 1 once (then restart the kernel), then Sections 2–8 top to bottom. Sections 11–12 are for files; 13–15 are live.

## 1. Environment (install in the VS Code terminal, then verify here)

Installing inside a notebook cell hides progress and can hang on multi-GB downloads. Run these in the VS Code terminal (Ctrl+`) from the project folder:

```bash
# Python 3.11 venv via uv (avoids missing-wheel problems on newer Python)
curl -LsSf https://astral.sh/uv/install.sh | sh && source ~/.local/bin/env
uv venv --seed --python 3.11 .venv
source .venv/bin/activate

# PyTorch: check "CUDA Version" in the top-right of nvidia-smi
#   13.0 or higher -> default wheel (CUDA 13.0)
python -m pip install torch torchaudio
#   12.x           -> use this instead
# python -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126

python -m pip install ipykernel transformers accelerate soundfile librosa sounddevice pandas \\
    "huggingface_hub<2.0,>=1.23.0" silero-vad onnxruntime
python -m pip install openwakeword
# if that fails on 'tflite-runtime':
# python -m pip install --no-deps openwakeword && python -m pip install tqdm scipy scikit-learn requests
```

Then in VS Code select the `.venv` kernel, **restart the kernel**, and run the cell below.

In [ ]:
# Section 1: VERIFY the environment. Install from the VS Code terminal (see markdown above), not here.
import importlib, sys
print("Python :", sys.version.split()[0], "|", sys.executable)
needed = ["torch", "transformers", "soundfile", "librosa", "sounddevice", "pandas",
          "huggingface_hub", "silero_vad", "openwakeword", "onnxruntime"]
missing = []
for name in needed:
    try:
        mod = importlib.import_module(name)
        print(f"  ok       {name:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:          # sounddevice raises OSError if PortAudio is missing
        missing.append(name)
        print(f"  MISSING  {name:16s} {type(e).__name__}: {str(e)[:90]}")
if "torch" not in missing:
    import torch
    print("CUDA available:", torch.cuda.is_available(), "| torch CUDA build:", torch.version.cuda)
    if not torch.cuda.is_available():
        print(">>> CUDA NOT AVAILABLE: reinstall torch from the terminal (see markdown), then restart the kernel.")
if missing:
    print(">>> Install the missing packages from the terminal, then restart the kernel.")
else:
    print("Environment OK. Continue with Section 2.")

## 2. Configuration
Every tunable value lives here. The endpointing values are starting points: calibrate them in your room with your microphone (Section 16).

In [ ]:
import os, sys, time, json, math, queue, threading, tempfile, collections
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

# ---------------- audio ----------------
SAMPLE_RATE   = 16000
VAD_BLOCK     = 512      # Silero VAD expects exactly 512 samples at 16 kHz (32 ms)
WAKE_BLOCK    = 1280     # openWakeWord frame size (80 ms)

# ---------------- endpointing (Silero VAD, replaces the RMS threshold) ----------------
VAD_ON         = 0.50    # speech probability that starts / sustains speech
VAD_OFF        = 0.35    # below this a block counts as silence (hysteresis)
START_BLOCKS   = 2       # consecutive speech blocks needed to open a segment (~64 ms)
HANGOVER_MS    = 350     # silence needed to close a segment (was 1200 ms)
PRE_ROLL_MS    = 320     # audio kept from before speech onset
MIN_SEGMENT_MS = 200     # ignore clicks / bumps
MAX_SEGMENT_S  = 6.0     # hard cap

# ---------------- arming ----------------
ARM_MODE       = "wakeword"   # "wakeword" | "always"  ("always" = bench testing only)
WAKE_WORD      = "hey_jarvis"
WAKE_THRESHOLD = 0.5
ARM_WINDOW_S   = 5.0          # after the wake word, one motion command is accepted within this window
# STOP is ALWAYS honoured, with or without the wake word.

# ---------------- ASR ----------------
ASR_BACKEND     = "hf"        # "hf" (official HF release) | "nemo_ctc_masked" (Section 7.2)
ASR_REPO        = "ARTPARK-IISc/SraVaani-1.0"
NEMO_CHECKPOINT = "SraVaani-nemo-checkpoint.nemo"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- optional Laya STOP fallback ----------------
USE_LAYA_STOP_FALLBACK = False
LAYA_STOP_MIN          = 0.80

# ---------------- logging ----------------
LOG_ROOT           = Path("logs")
SAVE_SEGMENT_AUDIO = True     # every segment is saved, so live sessions become evaluation data
LOG_ROOT.mkdir(exist_ok=True)

## 3. Hardware check

In [ ]:
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__, "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU         :", props.name, f"| {props.total_memory / 2**30:.1f} GiB VRAM")
else:
    print("!" * 70)
    print("CUDA NOT AVAILABLE: everything will run on the i7 CPU and latency will be")
    print("several times worse. Re-run the install cell (CUDA wheel) and restart.")
    print("!" * 70)

# Inputs have variable length; cuDNN autotuning would re-tune for every new length.
torch.backends.cudnn.benchmark = False
# Keep CPU threads modest: the CPU only runs VAD, wake word and feature extraction.
if DEVICE == "cuda":
    torch.set_num_threads(4)

## 4. Hugging Face login
SraVaani-1.0 is gated: accept its conditions on the model page first.

In [ ]:
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Paste your Hugging Face access token: ").strip()
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")

## 5. Text layer: script analysis and canonicalisation

This is the core fix for "SraVaani transcribes Hindi as another Indian language". Brahmic scripts share a parallel Unicode layout, so Bengali, Gurmukhi, Gujarati, Odia, Tamil, Telugu, Kannada and Malayalam letters fold onto Devanagari by a fixed offset. Devanagari is then romanised, and spelling variation is collapsed into phonetic keys. `আগে`, `आगे`, `aage` and `age` all become the same key.

Pure Python with no dependencies, deterministic, and sub-millisecond.

In [ ]:
# ======================= TEXT LAYER (script-invariant) =======================
import re, unicodedata
from collections import Counter

SCRIPT_RANGES = [
    ("Devanagari", 0x0900, 0x097F), ("Bengali", 0x0980, 0x09FF),
    ("Gurmukhi", 0x0A00, 0x0A7F), ("Gujarati", 0x0A80, 0x0AFF),
    ("Odia", 0x0B00, 0x0B7F), ("Tamil", 0x0B80, 0x0BFF),
    ("Telugu", 0x0C00, 0x0C7F), ("Kannada", 0x0C80, 0x0CFF),
    ("Malayalam", 0x0D00, 0x0D7F), ("OlChiki", 0x1C50, 0x1C7F),
    ("MeeteiMayek", 0xABC0, 0xABFF),
]
ALLOWED_SCRIPTS = {"Devanagari", "Latin"}

def char_script(ch):
    cp = ord(ch)
    if ("a" <= ch.lower() <= "z"):
        return "Latin"
    for name, lo, hi in SCRIPT_RANGES:
        if lo <= cp <= hi:
            return name
    return None if not ch.isalpha() and not unicodedata.category(ch).startswith("M") else "Other"

def analyze_scripts(text):
    """Histogram of scripts in a transcript + share of characters outside Hindi/English scripts."""
    counts = Counter(s for s in map(char_script, text) if s)
    total = sum(counts.values())
    foreign = sum(v for k, v in counts.items() if k not in ALLOWED_SCRIPTS)
    return {
        "scripts": dict(counts),
        "dominant_script": counts.most_common(1)[0][0] if counts else None,
        "foreign_fraction": (foreign / total) if total else 0.0,
        "script_violation": foreign > 0,
    }

# Brahmic scripts share the ISCII-derived layout: each block is 0x80 wide and
# mostly parallel to Devanagari, so most letters fold by a fixed offset.
_SPECIAL_FOLD = {
    "\u0A70": "\u0902",   # Gurmukhi tippi  -> anusvara
    "\u0A71": "",         # Gurmukhi addak  (gemination) -> drop
    "\u09CE": "\u0924\u094D",  # Bengali khanda ta -> त्
}
def fold_to_devanagari(text):
    out = []
    for ch in text:
        if ch in _SPECIAL_FOLD:
            out.append(_SPECIAL_FOLD[ch]); continue
        cp = ord(ch)
        if 0x0980 <= cp <= 0x0D7F:
            out.append(chr(cp - ((cp - 0x0900) // 0x80) * 0x80))
        else:
            out.append(ch)
    return "".join(out)

_CONS = {
    "क":"k","ख":"kh","ग":"g","घ":"gh","ङ":"n","च":"ch","छ":"chh","ज":"j","झ":"jh","ञ":"n",
    "ट":"t","ठ":"th","ड":"d","ढ":"dh","ण":"n","त":"t","थ":"th","द":"d","ध":"dh","न":"n","ऩ":"n",
    "प":"p","फ":"ph","ब":"b","भ":"bh","म":"m","य":"y","र":"r","ऱ":"r","ल":"l","ळ":"l","ऴ":"l",
    "व":"v","श":"sh","ष":"sh","स":"s","ह":"h",
}
_NUKTA = {"क":"q","ख":"kh","ग":"g","ज":"z","ड":"r","ढ":"rh","फ":"f","य":"y"}
_PRECOMPOSED_NUKTA = {"\u0958":"क","\u0959":"ख","\u095A":"ग","\u095B":"ज",
                      "\u095C":"ड","\u095D":"ढ","\u095E":"फ","\u095F":"य"}
_MATRA = {"ा":"a","ि":"i","ी":"i","ु":"u","ू":"u","ृ":"ri","ॄ":"ri","ॅ":"e","ॆ":"e","े":"e",
          "ै":"ai","ॉ":"o","ॊ":"o","ो":"o","ौ":"au"}
_VOWEL = {"अ":"a","आ":"a","इ":"i","ई":"i","उ":"u","ऊ":"u","ऋ":"ri","ॠ":"ri","ऌ":"li","ऍ":"e",
          "ऎ":"e","ए":"e","ऐ":"ai","ऑ":"o","ऒ":"o","ओ":"o","औ":"au"}
_SIGN = {"ं":"n","ँ":"n","ः":"h","ऽ":"","।":" ","॥":" "}
VIRAMA, NUKTA = "\u094D", "\u093C"

def _romanize_word(word):
    chars = []
    for ch in word:  # expand precomposed nukta letters into base + nukta
        if ch in _PRECOMPOSED_NUKTA:
            chars += [_PRECOMPOSED_NUKTA[ch], NUKTA]
        else:
            chars.append(ch)
    out, i, last_inherent = [], 0, False
    while i < len(chars):
        ch = chars[i]
        last_inherent = False
        if ch in _CONS:
            sound = _CONS[ch]
            if i + 1 < len(chars) and chars[i + 1] == NUKTA:
                sound = _NUKTA.get(ch, sound); i += 1
            out.append(sound)
            nxt = chars[i + 1] if i + 1 < len(chars) else ""
            if nxt in _MATRA:
                out.append(_MATRA[nxt]); i += 1
            elif nxt == VIRAMA:
                i += 1
            else:
                out.append("a"); last_inherent = True   # inherent schwa
        elif ch in _VOWEL:
            out.append(_VOWEL[ch])
        elif ch in _SIGN:
            out.append(_SIGN[ch])
        elif "\u0966" <= ch <= "\u096F":
            out.append(str(ord(ch) - 0x0966))
        elif ch.isascii():
            out.append(ch.lower())
        # anything else (unmapped marks/letters) is dropped
        i += 1
    if last_inherent and len(out) > 2:   # Hindi word-final schwa deletion
        out.pop()
    return "".join(out)

def romanize(text):
    """Any Indic script / Latin -> lowercase phonetic-ish roman text."""
    text = unicodedata.normalize("NFC", text or "")
    text = re.sub(r"<[^>]*>", " ", text)          # drop tags such as <pause>
    text = fold_to_devanagari(text)
    words = re.split(r"[\s\u0964\u0965,.;:!?\"()\[\]{}]+", text)
    return " ".join(w for w in (_romanize_word(w) for w in words) if w)

def phonetic_key(token):
    """Collapse spelling variation so 'aage'/'age'/'आगे' and 'peeche'/'pichhe'/'পিছে' meet."""
    t = re.sub(r"[^a-z0-9]", "", token.lower())
    t = t.replace("w", "v").replace("ph", "f").replace("z", "j").replace("q", "k")
    t = t.replace("ee", "i").replace("oo", "u")
    t = re.sub(r"(.)\1+", r"\1", t)                 # doubled letters
    t = re.sub(r"([kgcjtdpb])h", r"\1", t)          # drop aspiration
    t = re.sub(r"(.)\1+", r"\1", t)
    return t

def canonical_tokens(text):
    return [k for k in (phonetic_key(w) for w in romanize(text).split()) if k]

## 6. Command lexicon and parser

The command set is data, not code: extend `COMMAND_LEXICON` with the phrasing your users actually produce (Section 15 shows you what they say). Each phrase can be written in roman Hinglish, Devanagari, or English; all forms go through the same canonicaliser.

**Decision rules, in priority order:**
1. Any STOP phrase → **stop**.
2. A motion phrase near a negation ("mat", "nahi", "don't") → **stop**. The chair never moves on a negated command.
3. Exactly one motion intent → that intent.
4. More than one motion intent → **other** (ambiguous; ask the user to repeat).
5. No match → **other**.

Matching is longest-phrase-first, so "seedhe haath" (right) beats "seedhe" (forward). Fuzzy matching allows 0, 1 or 2 edits depending on word length, and only between words that start with the same sound.

Note that "bas" (enough/stop) also means "just" in Hindi: "bas aage chalo" will STOP. That is a deliberate bias toward the safe failure; remove "bas" if it proves too noisy.

In [ ]:
# =========================== COMMAND LEXICON =================================
INTENTS = ["forward", "backward", "left", "right", "stop", "faster", "slower", "other"]
MOTION_INTENTS = {"forward", "backward", "left", "right", "faster", "slower"}

# Each phrase is written the way users say it (roman Hinglish), the way SraVaani
# tends to spell it (Devanagari, incl. English loanwords), and plain English.
# Everything passes through the same canonicaliser, so script does not matter.
COMMAND_LEXICON = {
    "stop": [
        "stop", "stop it", "halt", "wait", "hold on", "brake",
        "ruko", "ruk", "ruk jao", "ruk ja", "rukiye", "roko", "rok do", "rok", "bas",
        "thamo", "tham jao", "theher", "thehro", "theher jao",
        "स्टॉप", "रुको", "रुक", "रुक जाओ", "रुकिए", "रोको", "रोक दो", "बस", "थमो", "ठहरो", "ठहर जाओ", "वेट",
    ],
    "forward": [
        "forward", "go forward", "ahead", "go ahead", "straight", "go straight",
        "aage", "aage chalo", "aage badho", "aage jao", "seedha", "seedhe", "sidha", "seedha chalo",
        "आगे", "आगे चलो", "आगे बढ़ो", "आगे जाओ", "सीधा", "सीधे", "फॉरवर्ड", "स्ट्रेट",
    ],
    "backward": [
        "back", "go back", "backward", "backwards", "reverse",
        "peeche", "piche", "pichhe", "peeche jao", "peeche chalo",
        "पीछे", "पीछे जाओ", "पीछे चलो", "बैक", "बैकवर्ड", "रिवर्स",
    ],
    "left": [
        "left", "turn left", "go left", "left side",
        "baayen", "bayen", "baaye", "baayein", "baayi taraf", "baayen mudo", "ulte haath", "ulta haath",
        "बाएं", "बाएँ", "बायें", "बाये", "बाईं", "बाएं मुड़ो", "बाईं तरफ", "उल्टे हाथ", "लेफ्ट", "लेफ्ट साइड",
    ],
    "right": [
        "right", "turn right", "go right", "right side",
        "daayen", "dayen", "daaye", "daayein", "daayi taraf", "daayen mudo", "seedhe haath", "sidhe haath",
        "दाएं", "दाएँ", "दायें", "दाये", "दाईं", "दाएं मुड़ो", "दाईं तरफ", "सीधे हाथ", "राइट", "राइट साइड",
    ],
    "faster": [
        "faster", "fast", "speed up", "speed badhao", "tez", "tez karo", "tej", "jaldi", "jaldi chalo",
        "तेज़", "तेज", "तेज़ करो", "फास्ट", "फास्टर", "स्पीड बढ़ाओ", "जल्दी",
    ],
    "slower": [
        "slow", "slower", "slow down", "dheere", "dhire", "dheere chalo", "aaram se", "speed kam karo",
        "धीरे", "धीरे चलो", "स्लो", "स्लो डाउन", "आराम से", "स्पीड कम करो",
    ],
}
NEGATION_WORDS = ["mat", "nahi", "nahin", "not", "dont", "don't", "no", "मत", "नहीं", "ना", "डोंट"]
NEGATION_WINDOW = 2   # tokens on either side of a motion phrase

def _build_phrase_table():
    table = []
    for intent, phrases in COMMAND_LEXICON.items():
        for p in phrases:
            keys = tuple(canonical_tokens(p))
            if keys:
                table.append((intent, p, keys))
    # longest phrase first, so 'seedhe haath' (right) wins over 'seedhe' (forward)
    table.sort(key=lambda r: (len(r[2]), sum(map(len, r[2]))), reverse=True)
    uniq, seen = [], set()
    for row in table:
        if (row[0], row[2]) not in seen:
            seen.add((row[0], row[2])); uniq.append(row)
    return uniq

PHRASE_TABLE = _build_phrase_table()
NEGATION_KEYS = {k for w in NEGATION_WORDS for k in canonical_tokens(w)}

def _edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def _tolerance(n):
    return 0 if n <= 3 else (1 if n <= 6 else 2)

def _token_match(tok, key):
    if tok == key:
        return 0
    if not tok or not key or tok[0] != key[0]:      # fuzzy only with same first sound
        return None
    d = _edit_distance(tok, key)
    return d if d <= _tolerance(min(len(tok), len(key))) else None

def parse_command(text):
    """Deterministic transcript -> intent. STOP has absolute priority."""
    tokens = canonical_tokens(text)
    used = [False] * len(tokens)
    matches = []
    for intent, phrase, keys in PHRASE_TABLE:
        n = len(keys)
        for s in range(len(tokens) - n + 1):
            if any(used[s:s + n]):
                continue
            dists = [_token_match(tokens[s + k], keys[k]) for k in range(n)]
            if all(d is not None for d in dists):
                for k in range(n):
                    used[s + k] = True
                score = 1.0 - sum(dists) / max(1, sum(map(len, keys)))
                matches.append({"intent": intent, "phrase": phrase, "start": s, "end": s + n, "score": round(score, 3)})
    neg_pos = [i for i, t in enumerate(tokens) if t in NEGATION_KEYS and not used[i]]
    for m in matches:
        m["negated"] = m["intent"] in MOTION_INTENTS and any(
            m["start"] - NEGATION_WINDOW <= p < m["end"] + NEGATION_WINDOW for p in neg_pos)

    result = {"tokens": tokens, "matches": matches}
    if any(m["intent"] == "stop" for m in matches):
        best = max((m for m in matches if m["intent"] == "stop"), key=lambda m: m["score"])
        return {**result, "intent": "stop", "reason": "stop_phrase", "score": best["score"]}
    if any(m["negated"] for m in matches):
        # "aage mat jao" / "don't go left": never move; treat as a stop request
        return {**result, "intent": "stop", "reason": "negated_motion", "score": 1.0}
    motion = {m["intent"] for m in matches}
    if len(motion) == 1:
        intent = motion.pop()
        best = max(m["score"] for m in matches)
        return {**result, "intent": intent, "reason": "phrase", "score": best}
    if len(motion) > 1:
        return {**result, "intent": "other", "reason": "ambiguous:" + "+".join(sorted(motion)), "score": 0.0}
    return {**result, "intent": "other", "reason": "no_match", "score": 0.0}

### 6.1 Text-layer self-test (no audio needed)

In [ ]:
# Offline tests: no audio, no GPU. Includes the SAME command transcribed in the
# wrong script (Bengali, Gurmukhi, Tamil, Odia, Gujarati), which is SraVaani's
# documented failure mode. All must pass before you go further.
SELF_TEST = [
    ("ruko", "stop"), ("रुको", "stop"), ("রুকো", "stop"), ("ਰੁਕੋ", "stop"), ("ருகோ", "stop"),
    ("ରୁକୋ", "stop"), ("રુકો", "stop"), ("प्लीज स्टॉप", "stop"), ("please stop", "stop"),
    ("बस बस", "stop"), ("रुक जाओ भाई", "stop"), ("don't stop", "stop"), ("<pause> रुको", "stop"),
    ("aage chalo", "forward"), ("आगे चलो", "forward"), ("আগে চলো", "forward"), ("go straight", "forward"),
    ("peeche jao", "backward"), ("পিছে", "backward"), ("बैक करो", "backward"),
    ("turn left", "left"), ("लेफ्ट मुड़ो", "left"), ("बाएं मुड़ो", "left"), ("left side", "left"),
    ("राइट", "right"), ("দাঁয়ে", "right"), ("सीधे हाथ मुड़ो", "right"), ("right side jao", "right"),
    ("थोड़ा तेज़ करो", "faster"), ("speed badhao", "faster"), ("धीरे चलो", "slower"), ("slow down", "slower"),
    ("aage mat jao", "stop"), ("don't go left", "stop"),
    ("left right", "other"), ("मेरा नाम राहुल है", "other"), ("hey jarvis", "other"),
    ("hey jarvis left", "left"), ("pack my bag", "other"), ("", "other"),
]
failures = 0
for text, want in SELF_TEST:
    got = parse_command(text)
    ok = got["intent"] == want
    failures += not ok
    if not ok:
        print(f"FAIL {text!r}: expected {want}, got {got['intent']} ({got['reason']}) tokens={got['tokens']}")
print(f"{len(SELF_TEST) - failures}/{len(SELF_TEST)} text-layer tests passed")
assert failures == 0, "Fix the lexicon/canonicaliser before running audio."

## 7. ASR backends
### 7.1 Default: the official Hugging Face release

In [ ]:
import soundfile as sf

def _hyp_text(h):
    if isinstance(h, str):
        return h
    if isinstance(h, (list, tuple)) and h:
        return _hyp_text(h[0])
    return getattr(h, "text", str(h))


class SraVaaniHF:
    """Official HF release (TorchScript FP16, trust_remote_code).

    Tries in-memory decoding through AutoProcessor + generate() (no disk I/O) and
    falls back to the documented file-path transcribe(). The working path is
    chosen once at warm-up. Use compare_paths() on a real clip to confirm the
    in-memory path returns the same text as the documented one."""
    name = "hf"

    def __init__(self, repo, token, device):
        from transformers import AutoModel
        self.device = device
        self.model = AutoModel.from_pretrained(repo, trust_remote_code=True, token=token).to(device).eval()
        self.processor = None
        try:
            from transformers import AutoProcessor
            self.processor = AutoProcessor.from_pretrained(repo, trust_remote_code=True, token=token)
        except Exception as e:
            print("AutoProcessor unavailable; using file-path transcribe():", repr(e)[:160])
        self._tmp = Path(tempfile.gettempdir()) / f"sravaani_seg_{os.getpid()}.wav"
        self.mode, self._fn = None, None

    def _via_processor(self, audio, half):
        inputs = self.processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        moved = {}
        for k, v in inputs.items():
            if torch.is_tensor(v):
                v = v.to(self.device)
                if half and v.is_floating_point():
                    v = v.half()
            moved[k] = v
        return self.processor.batch_decode(self.model.generate(**moved))[0]

    def _via_file(self, audio):
        sf.write(self._tmp, audio, SAMPLE_RATE, subtype="PCM_16")
        return _hyp_text(self.model.transcribe(str(self._tmp), return_hypotheses=True))

    def _candidates(self):
        c = []
        if self.processor is not None:
            c += [("processor", lambda a: self._via_processor(a, False)),
                  ("processor_fp16", lambda a: self._via_processor(a, True))]
        return c + [("file", self._via_file)]

    @torch.inference_mode()
    def select_mode(self, probe):
        for name, fn in self._candidates():
            try:
                if isinstance(fn(probe), str):
                    self.mode, self._fn = name, fn
                    print(f"SraVaaniHF decoding path: {name}")
                    return
            except Exception as e:
                print(f"  path '{name}' failed: {repr(e)[:160]}")
        raise RuntimeError("No working SraVaani decoding path")

    @torch.inference_mode()
    def compare_paths(self, audio):
        for name, fn in self._candidates():
            try:
                print(f"{name:15s}: {fn(audio)!r}")
            except Exception as e:
                print(f"{name:15s}: FAILED {repr(e)[:120]}")

    @torch.inference_mode()
    def __call__(self, audio):
        if self._fn is None:
            self.select_mode(audio)
        return {"text": self._fn(audio).strip(), "extras": {"decode_path": self.mode}}

### 7.2 Optional: script-constrained CTC decoding (NeMo checkpoint)

This backend enforces "Hindi + English only" *inside* the decoder. It runs SraVaani's encoder and its CTC head, then sets every output token that is not Devanagari or Latin to −∞ before the per-frame argmax. Bengali/Tamil/etc. output becomes impossible rather than merely corrected afterwards. It also skips the autoregressive TDT loop.

It also reports `unconstrained_drift`: the share of non-blank frames where the *unconstrained* model preferred a non-Hindi/English token. That number is your direct measurement of how often SraVaani was drifting into other languages.

**Requirements and caveats:**
- It needs the `.nemo` checkpoint, available on request via the SraVaani GitHub repo README.
- It needs NeMo: `pip install "nemo_toolkit[asr]"`. NeMo is far easier on Linux or WSL2 than on native Windows.
- CTC greedy decoding can be slightly less accurate than the default TDT decoding. Compare the two with Section 12 before switching.
- The attribute names used (`preprocessor`, `encoder`, `ctc_decoder`, `tokenizer`) are standard for NeMo hybrid TDT-CTC models. I could not inspect this specific checkpoint, so the constructor checks for them and fails loudly if any are missing.

In [ ]:
def _piece_allowed(piece):
    """True for subword pieces that are Devanagari, Latin, word-boundary, or tags."""
    p = piece.replace("\u2581", "")
    if p == "":
        return True
    if p.startswith("<") and p.endswith(">"):
        return p.lower() != "<unk>"
    return all(("\u0900" <= c <= "\u097F") or ("a" <= c <= "z") or c == "'" for c in p)


class SraVaaniNemoCTCMasked:
    name = "nemo_ctc_masked"

    def __init__(self, nemo_path, device):
        import nemo.collections.asr as nemo_asr
        self.device = device
        self.model = nemo_asr.models.ASRModel.restore_from(nemo_path, map_location=device).eval()
        for attr in ("preprocessor", "encoder", "ctc_decoder", "tokenizer"):
            if not hasattr(self.model, attr):
                raise RuntimeError(f"Checkpoint has no '{attr}'; it is not a hybrid TDT/RNNT-CTC model.")
        try:
            self.model.preprocessor.featurizer.dither = 0.0   # deterministic features
            self.model.preprocessor.featurizer.pad_to = 0
        except Exception:
            pass
        tok = self.model.tokenizer
        self.tokenizer = tok
        vocab = tok.vocab_size
        self.blank = vocab                                    # NeMo CTC blank = last index
        pieces = tok.ids_to_tokens(list(range(vocab)))
        allowed = [i for i, p in enumerate(pieces) if _piece_allowed(p)]
        self.allowed_ids = torch.tensor(allowed + [self.blank], device=device)
        print(f"Vocabulary: {vocab} pieces, {len(allowed)} allowed (Devanagari + Latin + specials)")

    @torch.inference_mode()
    def __call__(self, audio):
        sig = torch.from_numpy(audio).float().unsqueeze(0).to(self.device)
        length = torch.tensor([sig.shape[1]], device=self.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.device == "cuda"):
            feats, feat_len = self.model.preprocessor(input_signal=sig, length=length)
            enc, enc_len = self.model.encoder(audio_signal=feats, length=feat_len)
            logp = self.model.ctc_decoder(encoder_output=enc)
        logp = logp[0, : int(enc_len[0])].float()

        raw_best = logp.argmax(-1)
        masked = torch.full_like(logp, float("-inf"))
        masked[:, self.allowed_ids] = logp[:, self.allowed_ids]
        best = masked.argmax(-1)

        nonblank = raw_best != self.blank
        foreign = (~torch.isin(raw_best, self.allowed_ids)) & nonblank
        drift = foreign.sum().item() / max(1, nonblank.sum().item())

        ids, prev = [], None
        for t in best.tolist():
            if t != prev and t != self.blank:
                ids.append(t)
            prev = t
        return {"text": self.tokenizer.ids_to_text(ids).strip(),
                "extras": {"decode_path": "ctc_masked", "unconstrained_drift": round(drift, 3)}}

## 8. Load the ASR, warm it up, and measure latency

In [ ]:
def build_asr(backend=ASR_BACKEND):
    if backend == "nemo_ctc_masked":
        return SraVaaniNemoCTCMasked(NEMO_CHECKPOINT, DEVICE)
    return SraVaaniHF(ASR_REPO, HF_TOKEN, DEVICE)

t0 = time.perf_counter()
asr = build_asr()
print(f"Loaded {asr.name} on {DEVICE} in {time.perf_counter() - t0:.1f} s")

# Warm-up: the first CUDA calls are slow (kernel loading, allocator growth).
rng = np.random.default_rng(0)
def _noise(seconds):
    return (rng.standard_normal(int(seconds * SAMPLE_RATE)) * 0.003).astype(np.float32)

if hasattr(asr, "select_mode"):
    asr.select_mode(_noise(1.0))
for s in (0.8, 1.5, 3.0, 1.0):
    asr(_noise(s))
if DEVICE == "cuda":
    torch.cuda.synchronize()

# Latency by utterance length (median of 5). Plug the laptop in: mobile GPUs clock down on battery.
for s in (1.0, 2.0, 4.0):
    times = []
    for _ in range(5):
        a = _noise(s)
        t = time.perf_counter(); asr(a)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t) * 1000)
    print(f"ASR on {s:.1f} s audio: p50 {np.median(times):6.1f} ms   max {max(times):6.1f} ms")
if DEVICE == "cuda":
    print(f"Peak VRAM allocated by PyTorch: {torch.cuda.max_memory_allocated() / 2**20:.0f} MiB")

## 9. Optional: Laya as a STOP-only safety net (off by default)

In [ ]:
# Laya is NOT in the decision path. It can only ever ADD a stop: if the lexicon
# found nothing and Laya is confident the speaker wants to stop or wait, we stop.
# It can never cause motion. Default off. Result key names follow the published
# Router.predict shape; print one raw result and adjust if your version differs.
laya_router = None
LAYA_QUESTIONS = {
    "command": {
        "type": "choice",
        "instructions": "Is the speaker telling a wheelchair to stop, wait or halt?",
        "criteria": {
            "stop":  "stop, wait, halt, hold on, enough, ruko, ruk jao, bas, thehro, roko",
            "other": "anything else",
        },
    }
}

if USE_LAYA_STOP_FALLBACK:
    from laya import Router
    laya_router = Router(preload=True)
    _r = laya_router.predict({"utterance": "रुको"}, LAYA_QUESTIONS, model="multilingual")
    print("Laya warm-up routing:", _r.get("routing"))

def laya_stop_probability(text):
    res = laya_router.predict({"utterance": text}, LAYA_QUESTIONS, model="multilingual")
    ans = res.get("answers", {}).get("command", {})
    probs = ans.get("probabilities") or ans.get("distribution") or {}
    p = probs.get("stop")
    if p is None and ans.get("choice") == "stop":
        p = ans.get("confidence")
    return p, res.get("routing", {}).get("model")

## 10. Decision function: audio → intent

In [ ]:
def decide(audio, asr_backend=None):
    """audio (float32, 16 kHz mono) -> transcript -> intent, with per-stage timing."""
    asr_backend = asr_backend or asr
    t0 = time.perf_counter()
    out = asr_backend(audio)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    text = out["text"]
    parsed = parse_command(text)
    scripts = analyze_scripts(text)
    t2 = time.perf_counter()

    laya_info, laya_ms = None, 0.0
    if USE_LAYA_STOP_FALLBACK and laya_router is not None and parsed["intent"] == "other" and text:
        p, route = laya_stop_probability(text)
        laya_ms = (time.perf_counter() - t2) * 1000
        laya_info = {"p_stop": p, "routing": route}
        if p is not None and p >= LAYA_STOP_MIN:
            parsed = {**parsed, "intent": "stop", "reason": f"laya_stop_fallback({p:.2f})", "score": p}

    return {
        "transcript": text,
        "romanized": romanize(text),
        "tokens": parsed["tokens"],
        "intent": parsed["intent"],
        "reason": parsed["reason"],
        "score": parsed["score"],
        "matches": parsed["matches"],
        **scripts,
        "asr_extras": out.get("extras", {}),
        "laya": laya_info,
        "audio_s": round(len(audio) / SAMPLE_RATE, 3),
        "asr_ms": round((t1 - t0) * 1000, 1),
        "parse_ms": round((t2 - t1) * 1000, 2),
        "laya_ms": round(laya_ms, 1),
    }

def load_audio(path):
    """Any audio file -> float32 mono 16 kHz."""
    try:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        data = data.mean(axis=1)
    except Exception:
        import librosa
        data, sr = librosa.load(str(path), sr=None, mono=True)
    if sr != SAMPLE_RATE:
        import librosa
        data = librosa.resample(data, orig_sr=sr, target_sr=SAMPLE_RATE)
    return np.ascontiguousarray(data, dtype=np.float32)

## 11. Try one file

In [ ]:
TEST_AUDIO_PATH = "path/to/your/test_clip.wav"   # <-- change this

if os.path.exists(TEST_AUDIO_PATH):
    a = load_audio(TEST_AUDIO_PATH)
    r = decide(a)
    for k in ("transcript", "romanized", "tokens", "intent", "reason", "score",
              "dominant_script", "script_violation", "asr_extras", "audio_s", "asr_ms", "parse_ms"):
        print(f"{k:17s}: {r[k]}")
    if hasattr(asr, "compare_paths"):
        print("\nDecoding-path comparison (should be identical):")
        asr.compare_paths(a)
else:
    print(f"No file at {TEST_AUDIO_PATH}: update the path and re-run.")

## 12. Evaluation harness

Build a CSV with columns `filename,true_label` and, optionally, `reference` (what was actually said, in any script). Use the labels `forward, backward, left, right, stop, faster, slower, other`.

Include **negatives** labelled `other`: nearby conversation, TV, words that sound like commands. Without them you cannot measure false actuation, which is the number that matters most. Section 15 can generate a labelling sheet from your live sessions.

The report prioritises the metrics in this order:
1. STOP recall
2. False actuation (non-commands that would move the chair)
3. Wrong-motion (a different movement than asked)
4. Accuracy
5. Script violations
6. Latency
7. CER on canonical text

`compare_runs()` runs a paired McNemar test, so you can tell whether a change (for example `hf` vs `nemo_ctc_masked`) actually helped on the same clips.

In [ ]:
import pandas as pd
from IPython.display import display

LABELED_CSV       = "path/to/labels.csv"        # <-- change this
LABELED_AUDIO_DIR = Path("path/to/audio_folder")

def evaluate(csv_path, audio_dir, asr_backend=None, tag=None):
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    rows = []
    for _, r in df.iterrows():
        f = Path(audio_dir) / r["filename"]
        if not f.exists():
            print("missing:", f); continue
        res = decide(load_audio(f), asr_backend)
        row = {"file": r["filename"], "true": r["true_label"].strip().lower(), "pred": res["intent"],
               "reason": res["reason"], "transcript": res["transcript"],
               "dominant_script": res["dominant_script"], "script_violation": res["script_violation"],
               "audio_s": res["audio_s"], "asr_ms": res["asr_ms"],
               "drift": res["asr_extras"].get("unconstrained_drift")}
        ref = r.get("reference", "")
        if ref:
            ref_c, hyp_c = " ".join(canonical_tokens(ref)), " ".join(res["tokens"])
            row["cer"] = _edit_distance(hyp_c, ref_c) / max(1, len(ref_c))
        rows.append(row)
    out = pd.DataFrame(rows)
    name = tag or (asr_backend or asr).name
    path = LOG_ROOT / f"eval_{name}_{datetime.now():%Y%m%d_%H%M%S}.csv"
    out.to_csv(path, index=False, encoding="utf-8")
    print(f"{len(out)} clips evaluated, saved to {path}")
    return out

def report(df):
    n = len(df)
    is_motion_pred = df.pred.isin(MOTION_INTENTS)
    stop_rows = df[df.true == "stop"]
    other_rows = df[df.true == "other"]
    motion_rows = df[df.true.isin(MOTION_INTENTS)]
    summary = {
        "clips": n,
        "accuracy": (df.true == df.pred).mean(),
        "STOP_recall": (stop_rows.pred == "stop").mean() if len(stop_rows) else float("nan"),
        "false_actuation_rate (other->motion)":
            other_rows.pred.isin(MOTION_INTENTS).mean() if len(other_rows) else float("nan"),
        "wrong_motion_rate": ((motion_rows.pred != motion_rows.true) & motion_rows.pred.isin(MOTION_INTENTS)).mean()
            if len(motion_rows) else float("nan"),
        "nuisance_stop_rate (non-stop->stop)": (df[df.true != "stop"].pred == "stop").mean(),
        "script_violation_rate": df.script_violation.mean(),
        "asr_ms_p50": df.asr_ms.median(),
        "asr_ms_p95": df.asr_ms.quantile(0.95),
    }
    if "cer" in df:
        summary["canonical_CER"] = df.cer.mean()
    if df.drift.notna().any():
        summary["mean_unconstrained_drift"] = df.drift.mean()
    display(pd.Series(summary, name="summary").to_frame())

    print("Confusion matrix (rows = true, cols = predicted):")
    display(pd.crosstab(df.true, df.pred).reindex(index=INTENTS, columns=INTENTS, fill_value=0))

    per = []
    for l in INTENTS:
        tp = ((df.true == l) & (df.pred == l)).sum()
        fp = ((df.true != l) & (df.pred == l)).sum()
        fn = ((df.true == l) & (df.pred != l)).sum()
        per.append({"label": l, "precision": tp / (tp + fp) if tp + fp else float("nan"),
                    "recall": tp / (tp + fn) if tp + fn else float("nan"), "support": tp + fn})
    display(pd.DataFrame(per))

    misses = stop_rows[stop_rows.pred != "stop"]
    print(f"\nSTOP misses: {len(misses)}")
    if len(misses):
        display(misses[["file", "pred", "reason", "transcript", "dominant_script"]])
    fa = other_rows[other_rows.pred.isin(MOTION_INTENTS)]
    print(f"False actuations: {len(fa)}")
    if len(fa):
        display(fa[["file", "pred", "transcript"]])
    print("\nScript of transcripts:")
    display(df.dominant_script.value_counts(dropna=False))
    return summary

def compare_runs(a, b, name_a="A", name_b="B"):
    """Paired comparison on the same clips (exact McNemar test)."""
    m = a.merge(b, on="file", suffixes=("_a", "_b"))
    ca, cb = m.true_a == m.pred_a, m.true_b == m.pred_b
    only_a, only_b = int((ca & ~cb).sum()), int((~ca & cb).sum())
    k, n = min(only_a, only_b), only_a + only_b
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    print(f"{len(m)} paired clips | {name_a} acc {ca.mean():.3f} | {name_b} acc {cb.mean():.3f}")
    print(f"only {name_a} right: {only_a} | only {name_b} right: {only_b} | McNemar exact p = {p:.4f}")
    return m[ca != cb][["file", "true_a", "pred_a", "pred_b", "transcript_a", "transcript_b"]]

if os.path.exists(LABELED_CSV):
    eval_df = evaluate(LABELED_CSV, LABELED_AUDIO_DIR)
    eval_summary = report(eval_df)
else:
    print(f"Labels CSV not found at {LABELED_CSV}: create it first (see the markdown above).")

## 13. Real-time engine

The audio callback only enqueues. The main loop runs the wake word and VAD on the CPU (about 1 ms per block), and the worker thread runs ASR on the GPU.

**Gating:** STOP is always executed. A motion intent is executed only if the wake word fired within `ARM_WINDOW_S` before the segment ended; each wake word arms exactly one motion. You can say the wake word and the command in one breath ("hey jarvis, left") or with a pause.

**Interim STOP path:** there is no pretrained "ruko" keyword model, so for now STOP goes through VAD + ASR without needing the wake word. Expected time from end of speech to decision is roughly the hangover (~350 ms) plus GPU ASR time. The next step is a dedicated always-on STOP keyword spotter (Section 16).

In [ ]:
import sounddevice as sd

def load_vad():
    from silero_vad import load_silero_vad
    return load_silero_vad()

def load_wake_model(name):
    """Handles both current and older openWakeWord APIs. ONNX avoids tflite issues on Windows."""
    import openwakeword
    from openwakeword.model import Model
    try:
        from openwakeword.utils import download_models
        download_models(model_names=[name])
    except Exception:
        pass
    errors = []
    for kwargs in ({"wakeword_models": [name], "inference_framework": "onnx"},
                   {"wakeword_models": [name]}):
        try:
            return Model(**kwargs)
        except Exception as e:
            errors.append(repr(e)[:120])
    try:  # legacy API used by the original notebook
        return Model(wakeword_model_paths=[openwakeword.models[name]["model_path"]])
    except Exception as e:
        errors.append(repr(e)[:120])
    raise RuntimeError("Could not load wake word model: " + " | ".join(errors))


class SessionLog:
    def __init__(self):
        self.dir = LOG_ROOT / datetime.now().strftime("session_%Y%m%d_%H%M%S")
        (self.dir / "segments").mkdir(parents=True, exist_ok=True)
        self._f = open(self.dir / "events.jsonl", "a", encoding="utf-8")
        self._lock = threading.Lock()

    def write(self, rec):
        with self._lock:
            self._f.write(json.dumps(rec, ensure_ascii=False, default=float) + "\n")
            self._f.flush()

    def save_audio(self, seg_id, audio):
        name = f"seg_{seg_id:05d}.wav"
        sf.write(self.dir / "segments" / name, audio, SAMPLE_RATE, subtype="PCM_16")
        return name

    def close(self):
        self._f.close()


class ConsoleSink:
    """Final output today. Swap for a serial/CAN sink when a motor controller exists."""
    def emit(self, action):
        print(f">>> [{action['time']}] {action['intent'].upper():9s} "
              f"'{action['transcript']}' -> {action['romanized']!r} | {action['reason']} "
              f"| speech-end->decision {action['latency_ms']['speech_end_to_decision']:.0f} ms")

# class SerialSink:   # future: motor controller over UART
#     def __init__(self, port="COM5", baud=115200):
#         import serial; self.ser = serial.Serial(port, baud, timeout=0.1)
#     def emit(self, action):
#         self.ser.write((json.dumps({"intent": action["intent"], "id": action["seg_id"]}) + "\n").encode())


class RealtimeEngine:
    """mic -> Silero VAD endpointing (+ wake word) -> ASR worker thread -> parser -> gate -> sink.

    The audio loop never blocks on inference, so listening, and therefore STOP,
    continues while a segment is being transcribed."""

    def __init__(self, asr_backend, sink=None, arm_mode=ARM_MODE):
        self.asr = asr_backend
        self.sink = sink or ConsoleSink()
        self.vad = load_vad()
        self.arm_mode = arm_mode
        self.wake = None
        if arm_mode == "wakeword":
            try:
                self.wake = load_wake_model(WAKE_WORD)
            except Exception as e:
                print("Wake word unavailable:", e)
                print(">>> Falling back to ARM_MODE='always' (acceptable only while no motor is connected).")
                self.arm_mode = "always"
        self.audio_q, self.seg_q = queue.Queue(), queue.Queue()
        self._arm_lock = threading.Lock()
        self.armed_at = None
        self.seg_counter = 0
        self.log = None

    def _callback(self, indata, frames, t, status):
        if status:
            self.audio_q.put(("status", str(status)))
        self.audio_q.put(("audio", indata[:, 0].copy()))

    def _consume_arm(self, token):
        with self._arm_lock:
            if token is not None and self.armed_at == token:
                self.armed_at = None
                return True
            return False

    def _worker(self):
        while True:
            item = self.seg_q.get()
            if item is None:
                break
            seg_id, audio, t_speech_end, t_closed, arm_token = item
            backlog = self.seg_q.qsize()
            try:
                res = decide(audio, self.asr)
            except Exception as e:
                print(f"[seg {seg_id}] pipeline error: {e!r}")
                self.log.write({"seg_id": seg_id, "error": repr(e)})
                continue
            t_dec = time.perf_counter()
            intent = res["intent"]
            if intent == "stop":
                executed, gate = True, "stop_always_allowed"
            elif intent in MOTION_INTENTS:
                if self.arm_mode == "always":
                    executed, gate = True, "always_armed"
                elif self._consume_arm(arm_token):
                    executed, gate = True, "armed_by_wake_word"
                else:
                    executed, gate = False, "not_armed"
            else:
                executed, gate = False, "no_command"

            latency = {"endpoint_hangover": (t_closed - t_speech_end) * 1000,
                       "queue_wait": max(0.0, (t_dec - t_closed) * 1000 - res["asr_ms"] - res["parse_ms"] - res["laya_ms"]),
                       "asr": res["asr_ms"], "parse": res["parse_ms"], "laya": res["laya_ms"],
                       "speech_end_to_decision": (t_dec - t_speech_end) * 1000}
            action = {"time": datetime.now().strftime("%H:%M:%S"), "seg_id": seg_id, "intent": intent,
                      "reason": res["reason"], "transcript": res["transcript"], "romanized": res["romanized"],
                      "latency_ms": latency}
            if executed:
                self.sink.emit(action)
            else:
                print(f"    [{action['time']}] ({gate}) heard '{res['transcript']}' -> {intent} ({res['reason']})")

            rec = {"seg_id": seg_id, "wall_time": datetime.now().isoformat(timespec="milliseconds"),
                   "executed": executed, "gate": gate, "backlog": backlog,
                   **{k: v for k, v in res.items()}, "latency_ms": latency}
            if SAVE_SEGMENT_AUDIO:
                rec["audio_file"] = self.log.save_audio(seg_id, audio)
            self.log.write(rec)

    def run(self, max_seconds=None, input_device=None):
        self.log = SessionLog()
        worker = threading.Thread(target=self._worker, daemon=True)
        worker.start()

        block_ms = VAD_BLOCK / SAMPLE_RATE * 1000
        hang_blocks = int(math.ceil(HANGOVER_MS / block_ms))
        max_blocks = int(MAX_SEGMENT_S * 1000 / block_ms)
        pre_roll = collections.deque(maxlen=max(1, int(PRE_ROLL_MS / block_ms)))
        state, seg, speech_run, silence_run, silence_started = "idle", [], 0, 0, None
        wake_buf = np.zeros(0, dtype=np.int16)
        t_start = time.time()

        mode = f"wake word '{WAKE_WORD}'" if self.arm_mode == "wakeword" else "ALWAYS ARMED (testing)"
        print(f"Listening | motion needs {mode} | STOP always active | logs: {self.log.dir}")
        print("Interrupt the kernel to stop.\n")

        with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="int16",
                            blocksize=VAD_BLOCK, callback=self._callback, device=input_device):
            try:
                while not (max_seconds and time.time() - t_start > max_seconds):
                    try:
                        kind, block = self.audio_q.get(timeout=0.5)
                    except queue.Empty:
                        continue
                    if kind == "status":
                        print("Audio status:", block); continue
                    now = time.perf_counter()

                    # --- wake word (CPU, 80 ms frames) ---
                    if self.wake is not None:
                        wake_buf = np.concatenate([wake_buf, block])
                        while len(wake_buf) >= WAKE_BLOCK:
                            frame, wake_buf = wake_buf[:WAKE_BLOCK], wake_buf[WAKE_BLOCK:]
                            scores = self.wake.predict(frame)
                            score = max(scores.values()) if scores else 0.0
                            if score >= WAKE_THRESHOLD:
                                with self._arm_lock:
                                    fresh = self.armed_at is None or now - self.armed_at > 1.0
                                    self.armed_at = now
                                if fresh:
                                    print(f"[wake] armed (score {score:.2f}) for {ARM_WINDOW_S:.0f} s")
                                try:
                                    self.wake.reset()
                                except Exception:
                                    pass

                    # --- VAD endpointing (CPU, 32 ms blocks) ---
                    prob = self.vad(torch.from_numpy(block.astype(np.float32) / 32768.0), SAMPLE_RATE).item()
                    if state == "idle":
                        pre_roll.append(block)
                        speech_run = speech_run + 1 if prob >= VAD_ON else 0
                        if speech_run >= START_BLOCKS:
                            state, seg, silence_run = "speech", list(pre_roll), 0
                        continue

                    seg.append(block)
                    if prob < VAD_OFF:
                        if silence_run == 0:
                            silence_started = now
                        silence_run += 1
                    else:
                        silence_run = 0
                    endpointed = silence_run >= hang_blocks
                    if not (endpointed or len(seg) >= max_blocks):
                        continue

                    audio = np.concatenate(seg).astype(np.float32) / 32768.0
                    if endpointed:   # trim the hangover silence, keep 100 ms tail
                        audio = audio[: max(1, len(audio) - silence_run * VAD_BLOCK + SAMPLE_RATE // 10)]
                    if len(audio) / SAMPLE_RATE * 1000 >= MIN_SEGMENT_MS:
                        with self._arm_lock:
                            armed = self.armed_at is not None and now - self.armed_at <= ARM_WINDOW_S
                            token = self.armed_at if armed else None
                        self.seg_counter += 1
                        speech_end = silence_started if endpointed else now
                        self.seg_q.put((self.seg_counter, audio, speech_end, now, token))
                    state, seg, speech_run, silence_run = "idle", [], 0, 0
                    pre_roll.clear()
                    self.vad.reset_states()
            except KeyboardInterrupt:
                print("\nStopped by user.")
            finally:
                self.seg_q.put(None)
                worker.join(timeout=15)
                self.log.close()
                print("Session saved to", self.log.dir)

## 14. Run live

In [ ]:
print(sd.query_devices())
INPUT_DEVICE = None      # set to an index from the list above if the default mic is wrong

engine = RealtimeEngine(asr)
engine.run(max_seconds=120, input_device=INPUT_DEVICE)   # max_seconds=None to run until interrupted

## 15. Session report and labelling sheet
Turn each live session into evaluation data: label `labels_todo.csv`, then feed it to Section 12.

In [ ]:
def session_report(session_dir=None):
    session_dir = Path(session_dir or engine.log.dir)
    ev = pd.read_json(session_dir / "events.jsonl", lines=True)
    ev = ev[ev.get("error").isna()] if "error" in ev else ev
    if ev.empty:
        print("No segments in this session."); return ev
    lat = pd.json_normalize(ev["latency_ms"])
    print(f"{len(ev)} segments | executed {int(ev.executed.sum())} | script violations {ev.script_violation.mean():.1%}")
    display(ev.groupby(["intent", "gate"]).size().rename("count").to_frame())
    display(lat.describe(percentiles=[0.5, 0.95]).loc[["50%", "95%", "max"]].round(1))
    print("Transcript scripts:")
    display(ev.dominant_script.value_counts(dropna=False))
    return ev

def export_labeling_sheet(session_dir=None):
    """Writes labels_todo.csv next to the segments; fill true_label (and reference), then use Section 12."""
    session_dir = Path(session_dir or engine.log.dir)
    ev = pd.read_json(session_dir / "events.jsonl", lines=True)
    ev = ev[ev.get("audio_file").notna()] if "audio_file" in ev else ev
    sheet = pd.DataFrame({"filename": ev.audio_file, "predicted": ev.intent,
                          "transcript": ev.transcript, "true_label": "", "reference": ""})
    path = session_dir / "segments" / "labels_todo.csv"
    sheet.to_csv(path, index=False, encoding="utf-8-sig")
    print("Labelling sheet:", path)
    print("Evaluate later with: evaluate(<filled csv>, Path(session_dir) / 'segments')")
    return path

_ev = session_report()
export_labeling_sheet()

## 16. Tuning notes and next steps

**Endpointing:**
- If commands get cut off, raise `HANGOVER_MS` (try 450) or lower `VAD_OFF`.
- If the end of speech is detected too late, lower `HANGOVER_MS`.
- If a cough or bump opens a segment, raise `START_BLOCKS` or `MIN_SEGMENT_MS`.

**Wake word:** lower `WAKE_THRESHOLD` if it misses; raise it on false wakes. "hey_jarvis" is a placeholder: train a custom phrase (openWakeWord supports this) that suits your users, or replace it with a physical button.

**Laptop power:** run on AC power with the Windows power mode set to *Best performance*. On battery, the A500 and the i7-1355U clock down and latency rises noticeably. Compare the latency printout from Section 8 on battery versus AC.

**Reading the logs:**
- `script_violation` / `dominant_script` show how often SraVaani leaves Hindi/English. With the default HF backend those transcripts are still handled by canonicalisation.
- With the NeMo backend, `unconstrained_drift` quantifies the drift directly, and masking prevents it.
- `latency_ms.queue_wait` greater than 0 means the worker is falling behind.

**Next steps, in order:**
1. Collect and label 1–2 hours of real sessions (Section 15). Evaluate with Section 12. Extend the lexicon from STOP misses and `no_match` transcripts.
2. Get the `.nemo` checkpoint and A/B test `nemo_ctc_masked` against `hf` with `compare_runs()`.
3. Train a dedicated always-on STOP keyword spotter (openWakeWord custom model on "ruko / roko / bas / stop") from the logged segments. Run it on every audio block, without VAD or ASR.
4. If accuracy is still short, do a decoder-only SraVaani fine-tune on your labelled segments (the SraVaani repo's `finetune.py` freezes the encoder by default).
5. When a motor controller exists, replace `ConsoleSink` with a serial/CAN sink. The microcontroller must own speed limits, ramps, a heartbeat watchdog, and the hardware E-stop. This notebook only *requests* motion.